In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)


# Remplace par le chemin réel vers ton fichier Parquet
chemin_fichier = 'data/domains/domains_US.parquet'

# Lecture du fichier Parquet
df = pd.read_parquet(chemin_fichier)
df = df.sort_values(by='domain')  # Tri par nom de domaine pour une meilleure lisibilité

# Affichage du DataFrame (Jupyter le rendra sous forme de beau tableau interactif)
display(df)  # Tri par nom de domaine pour une meilleure lisibilité

# Optionnel : si ton fichier est énorme, affiche juste les 5 premières lignes
# display(df.head())

,id,domain
426,103308,24hourhiphop.com
784,276336,404media.co
1216,74622,48hills.org
1522,254826,7al.net
1067,5498,abc27.com
1469,44765,abcnews.com
1354,4218,aberdeennews.com
356,91627,abilene-rc.com
118,2000,abqjournal.com
399,192238,accesswire.com


In [9]:
import duckdb

# Chemin vers ton fichier Parquet brut
chemin_parquet = 'gdelt_parquet_db/gdelt_2015-02.parquet'

# On définit la limite pour la vérification
min_themes = 1 

# Requête de vérification sur les 5 premières lignes
query = f"""
SELECT 
    m.GKGRECORDID,
    strptime(substr(CAST(m.DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE AS period,
    CAST(m.Tone AS DOUBLE) AS tone,
    SIGN(CAST(m.Tone AS DOUBLE)) AS tone_bin,
    ARRAY_LENGTH(string_split(m.EnhancedThemes, ';')) AS total_themes_count,
    list_transform(
        string_split(m.EnhancedThemes, ';'),
        x -> upper(trim(split_part(trim(x), ',', 1)))
    ) AS themes_list
FROM '{chemin_parquet}' m
WHERE ARRAY_LENGTH(string_split(m.EnhancedThemes, ';')) >= {min_themes}
LIMIT 5
"""

# Exécution et affichage
resultat = duckdb.query(query).to_df()
display(resultat)

,GKGRECORDID,period,tone,tone_bin,total_themes_count,themes_list
0,20150219071500-0,2015-02-19,-4.644531,-1,86,"[KILL, SEIGE, CHECKPOINT, UNREST_CHECKPOINT, W..."
1,20150219020000-0,2015-02-19,-2.222656,-1,26,"[CRIME_COMMON_ROBBERY, CRIME_COMMON_ROBBERY, D..."
2,20150219020000-1,2015-02-19,-2.068359,-1,21,"[TRAFFIC, DISCRIMINATION, ARREST, TRIAL, TAX_F..."
3,20150219020000-2,2015-02-19,-2.250000,-1,14,"[GENERAL_GOVERNMENT, ECON_INTEREST_RATES, ECON..."
4,20150219020000-3,2015-02-19,-1.104492,-1,44,"[TAX_WORLDLANGUAGES, TAX_FNCACT, TAX_FNCACT, T..."


In [23]:
import duckdb
import pandas as pd

# 1. Connexion et configuration minimale
con = duckdb.connect()

# 2. Création de la structure de test (simulant la table gkg_clean)
# On utilise tes propres fonctions de transformation pour garantir la cohérence
query_setup = """
CREATE TABLE gkg_clean AS 
SELECT 
    GKGRECORDID,
    strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE AS period,
    CAST(Tone AS DOUBLE) AS tone,
    SIGN(CAST(Tone AS DOUBLE)) AS tone_bin,
    ARRAY_LENGTH(string_split(EnhancedThemes, ';')) AS total_themes_count,
    list_transform(
        string_split(EnhancedThemes, ';'),
        x -> upper(trim(split_part(trim(x), ',', 1)))
    ) AS themes_list
FROM read_parquet('gdelt_parquet_db/gdelt_2015-02.parquet')
LIMIT 5;
"""
con.execute(query_setup)

# 3. Ton code de matching (copié depuis ton script)
# On enregistre une table de thèmes bidon pour tester le JOIN
sector_themes = pd.DataFrame({
    'cat_key': ['GENERAL_GOVERNMENT', 'TAX_WORLDLANGUAGES', 'CHECKPOINT'],
    'theme': ['GENERAL_GOVERNMENT', 'TAX_WORLDLANGUAGES', 'CHECKPOINT'] # Remplace par tes thèmes réels
})
con.register("sector_themes_tbl", sector_themes)

query_matching = """
WITH 
-- 1. Unnesting propre grâce à la liste pré-calculée dans gkg_clean
unnested AS (
    SELECT GKGRECORDID, period, tone, tone_bin, total_themes_count, unnest(themes_list) as theme
    FROM gkg_clean
),

-- 2. Matching exact avec les thèmes du secteur
matched AS (
    SELECT 
        u.GKGRECORDID, u.period, u.tone, u.tone_bin, u.total_themes_count,
        st.cat_key,
        COUNT(*) AS theme_hits
    FROM unnested u
    INNER JOIN sector_themes_tbl st ON u.theme = st.theme
    GROUP BY 1, 2, 3, 4, 5, 6
)

SELECT * FROM matched
"""

# 4. Affichage du résultat
resultat = con.execute(query_matching).df()
display(resultat)

,GKGRECORDID,period,tone,tone_bin,total_themes_count,cat_key,theme_hits
0,20150219071500-0,2015-02-19,-4.644531,-1,86,TAX_WORLDLANGUAGES,2
1,20150219020000-1,2015-02-19,-2.068359,-1,21,TAX_WORLDLANGUAGES,1
2,20150219071500-0,2015-02-19,-4.644531,-1,86,CHECKPOINT,1
3,20150219020000-2,2015-02-19,-2.250000,-1,14,GENERAL_GOVERNMENT,1
4,20150219020000-3,2015-02-19,-1.104492,-1,44,TAX_WORLDLANGUAGES,1


In [25]:
import duckdb
import pandas as pd

# Désactiver la limitation de largeur de colonne dans Jupyter
pd.set_option('display.max_colwidth', None)

con = duckdb.connect()

# 1. On récupère aussi la colonne originale 'EnhancedThemes' dans gkg_clean
query_setup = """
CREATE TABLE gkg_clean AS 
SELECT 
    GKGRECORDID,
    EnhancedThemes, -- On garde la version brute
    strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE AS period,
    CAST(Tone AS DOUBLE) AS tone,
    SIGN(CAST(Tone AS DOUBLE)) AS tone_bin,
    ARRAY_LENGTH(string_split(EnhancedThemes, ';')) AS total_themes_count,
    list_transform(
        string_split(EnhancedThemes, ';'),
        x -> upper(trim(split_part(trim(x), ',', 1)))
    ) AS themes_list
FROM read_parquet('gdelt_parquet_db/gdelt_2015-02.parquet')
LIMIT 5;
"""
con.execute(query_setup)

# 2. Matching en incluant EnhancedThemes dans les colonnes transportées
sector_themes = pd.DataFrame({
    'cat_key': ['GENERAL_GOVERNMENT', 'TAX_WORLDLANGUAGES', 'CHECKPOINT'],
    'theme': ['GENERAL_GOVERNMENT', 'TAX_WORLDLANGUAGES', 'CHECKPOINT'] # Remplace par tes thèmes réels
})
con.register("sector_themes_tbl", sector_themes)

query_matching = """
WITH 
unnested AS (
    -- On ajoute EnhancedThemes ici pour le garder dans le résultat final
    SELECT GKGRECORDID, EnhancedThemes, period, tone, tone_bin, total_themes_count, unnest(themes_list) as theme
    FROM gkg_clean
),

matched AS (
    SELECT 
        u.GKGRECORDID, u.EnhancedThemes, u.period, u.tone, u.tone_bin, u.total_themes_count,
        st.cat_key,
        COUNT(*) AS theme_hits
    FROM unnested u
    INNER JOIN sector_themes_tbl st ON u.theme = st.theme
    GROUP BY 1, 2, 3, 4, 5, 6, 7
)

SELECT * FROM matched
"""

resultat = con.execute(query_matching).df()
display(resultat)

,GKGRECORDID,EnhancedThemes,period,tone,tone_bin,total_themes_count,cat_key,theme_hits
0,20150219071500-0,"KILL,464;SEIGE,3001;CHECKPOINT,3001;UNREST_CHECKPOINT,3001;WOUND,511;MILITARY,1502;TAX_MILITARY_TITLE,1502;TAX_FNCACT,1502;TAX_FNCACT,1805;TAX_WORLDLANGUAGES,841;TAX_WORLDLANGUAGES,866;SOC_SUICIDE,229;SOC_SUICIDE,397;SOC_SUICIDE,739;SOC_SUICIDE,1586;SOC_SUICIDE,1900;SOC_SUICIDE,2328;SOC_SUICIDE,2664;TAX_FNCACT,2099;TAX_FNCACT,188;TAX_FNCACT,168;ARMEDCONFLICT,574;TAX_FNCACT,1246;SOC_POINTSOFINTEREST,45;SOC_POINTSOFINTEREST,141;SOC_POINTSOFINTEREST,270;SOC_POINTSOFINTEREST,342;SOC_POINTSOFINTEREST,373;SOC_POINTSOFINTEREST,454;SOC_POINTSOFINTEREST,659;SOC_POINTSOFINTEREST,787;SOC_POINTSOFINTEREST,1066;SOC_POINTSOFINTEREST,1233;SOC_POINTSOFINTEREST,1761;SOC_POINTSOFINTEREST,2140;SOC_POINTSOFINTEREST,2262;SOC_POINTSOFINTEREST,2410;TAX_WEAPONS,2334;TAX_WEAPONS,1119;PROTEST,761;TAX_WEAPONS,1593;TAX_WEAPONS,2671;SECURITY_SERVICES,32;SECURITY_SERVICES,159;SECURITY_SERVICES,257;SECURITY_SERVICES,360;SECURITY_SERVICES,415;SECURITY_SERVICES,646;SECURITY_SERVICES,774;SECURITY_SERVICES,857;SECURITY_SERVICES,879;SECURITY_SERVICES,1053;SECURITY_SERVICES,1220;SECURITY_SERVICES,2295;SECURITY_SERVICES,2397;SECURITY_SERVICES,2799;SECURITY_SERVICES,2851;SECURITY_SERVICES,2913;TAX_FNCACT,32;TAX_FNCACT,159;TAX_FNCACT,257;TAX_FNCACT,360;TAX_FNCACT,415;TAX_FNCACT,646;TAX_FNCACT,774;TAX_FNCACT,857;TAX_FNCACT,879;TAX_FNCACT,1053;TAX_FNCACT,1220;TAX_FNCACT,2295;TAX_FNCACT,2397;TAX_FNCACT,2799;TAX_FNCACT,2851;TAX_FNCACT,2913;TAX_FNCACT,2420;TERROR,237;TERROR,405;TERROR,1908;SUICIDE_ATTACK,237;SUICIDE_ATTACK,405;SUICIDE_ATTACK,1908;TAX_WEAPONS,237;TAX_WEAPONS,405;TAX_WEAPONS,1908;TAX_FNCACT,715;",2015-02-19,-4.644531,-1,86,CHECKPOINT,1
1,20150219020000-1,"TRAFFIC,986;DISCRIMINATION,569;ARREST,1012;TRIAL,9;TAX_FNCACT,9;TAX_MILITARY_TITLE,917;TAX_FNCACT,917;SECURITY_SERVICES,191;SECURITY_SERVICES,376;SECURITY_SERVICES,605;SECURITY_SERVICES,1144;SECURITY_SERVICES,1177;TAX_FNCACT,191;TAX_FNCACT,376;TAX_FNCACT,605;TAX_FNCACT,1144;TAX_FNCACT,1177;TAX_MILITARY_TITLE,294;TAX_FNCACT,294;TAX_WORLDLANGUAGES,753;",2015-02-19,-2.068359,-1,21,TAX_WORLDLANGUAGES,1
2,20150219020000-3,"TAX_WORLDLANGUAGES,2843;TAX_FNCACT,3153;TAX_FNCACT,3408;TAX_FNCACT,3436;TAX_FNCACT,3096;TAX_FNCACT,3336;EDUCATION,1018;EDUCATION,1126;EDUCATION,2646;SOC_POINTSOFINTEREST,1018;SOC_POINTSOFINTEREST,1126;SOC_POINTSOFINTEREST,2646;TAX_WORLDMAMMALS,2086;ENV_OIL,2975;LEADER,1057;LEADER,2587;TAX_FNCACT,1057;TAX_FNCACT,2587;TAX_FNCACT,2891;MEDIA_SOCIAL,2516;TAX_FNCACT,3618;TAX_FNCACT,3048;TAX_WORLDFISH,2202;TAX_POLITICAL_PARTY,354;SOC_POINTSOFINTEREST,1888;SOC_POINTSOFINTEREST,3781;TAX_FNCACT,1286;TAX_FNCACT,1346;TAX_FNCACT,48;TAX_FNCACT,132;TAX_FNCACT,570;TAX_FNCACT,623;TAX_FNCACT,988;TAX_FNCACT,2463;TAX_FNCACT,2600;SECURITY_SERVICES,1323;SECURITY_SERVICES,1776;TAX_FNCACT,1323;TAX_FNCACT,1776;EXTREMISM,2539;TAX_MILITARY_TITLE,3637;TAX_FNCACT,3637;TAX_FNCACT,1683;",2015-02-19,-1.104492,-1,44,TAX_WORLDLANGUAGES,1
3,20150219020000-2,"GENERAL_GOVERNMENT,2276;ECON_INTEREST_RATES,2246;ECON_STOCKMARKET,1857;TAX_WORLDMAMMALS,1601;TAX_WORLDMAMMALS,1700;TAX_WORLDMAMMALS,2121;KILL,39;KILL,1297;TAX_FNCACT,95;LEADER,107;TAX_FNCACT,107;MEDIA_MSM,832;ECON_DEBT,1948;",2015-02-19,-2.250000,-1,14,GENERAL_GOVERNMENT,1
4,20150219071500-0,"KILL,464;SEIGE,3001;CHECKPOINT,3001;UNREST_CHECKPOINT,3001;WOUND,511;MILITARY,1502;TAX_MILITARY_TITLE,1502;TAX_FNCACT,1502;TAX_FNCACT,1805;TAX_WORLDLANGUAGES,841;TAX_WORLDLANGUAGES,866;SOC_SUICIDE,229;SOC_SUICIDE,397;SOC_SUICIDE,739;SOC_SUICIDE,1586;SOC_SUICIDE,1900;SOC_SUICIDE,2328;SOC_SUICIDE,2664;TAX_FNCACT,2099;TAX_FNCACT,188;TAX_FNCACT,168;ARMEDCONFLICT,574;TAX_FNCACT,1246;SOC_POINTSOFINTEREST,45;SOC_POINTSOFINTEREST,141;SOC_POINTSOFINTEREST,270;SOC_POINTSOFINTEREST,342;SOC_POINTSOFINTEREST,373;SOC_POINTSOFINTEREST,454;SOC_POINTSOFINTEREST,659;SOC_POINTSOFINTEREST,787;SOC_POINTSOFINTEREST,1066;SOC_POINTSOFINTEREST,1233;SOC_POINTSOFINTEREST,1761;SOC_POINTSOFINTEREST,2140;SOC

In [34]:
import duckdb
import pandas as pd
from pathlib import Path

def explore_gdelt_locations(parquet_file_path: str, limit: int = 20):
    """
    Explore et compte les localisations (Pays et Villes) contenues dans un fichier GDELT brut.
    
    Paramètres:
    - parquet_file_path: Chemin vers un fichier parquet brut (ex: './gdelt_parquet_db_test/gdelt_202001.parquet')
    - limit: Nombre de résultats à afficher
    """
    
    # On se connecte en mémoire vive pour cette petite exploration
    con = duckdb.connect(database=':memory:')
    
    # La requête magique qui déplie et parse la chaîne complexe de GDELT
    query = f"""
    WITH unnested_locs AS (
        -- 1. On sépare chaque bloc de localisation (découpé par ;)
        SELECT unnest(string_split(EnhancedLocations, ';')) AS loc_str
        FROM read_parquet('{parquet_file_path}')
        WHERE EnhancedLocations IS NOT NULL AND EnhancedLocations != ''
    ),
    parsed_locs AS (
        -- 2. On découpe l'intérieur du bloc (découpé par #)
        SELECT 
            split_part(loc_str, '#', 1) AS LocType,
            split_part(loc_str, '#', 2) AS LocName,
            split_part(loc_str, '#', 3) AS CountryCode
        FROM unnested_locs
        WHERE loc_str != ''
    )
    -- 3. On compte et on trie
    SELECT 
        LocType,
        CountryCode,
        LocName,
        COUNT(*) AS mentions
    FROM parsed_locs
    WHERE CountryCode != ''
    GROUP BY LocType, CountryCode, LocName
    ORDER BY mentions DESC
    LIMIT {limit};
    """
    
    print(f"Analyse des localisations en cours sur {parquet_file_path} ...")
    df_locs = con.execute(query).df()
    
    # Petit dictionnaire pour rendre le LocType lisible
    type_map = {
        '1': 'Pays', 
        '2': 'État US', 
        '3': 'Ville US', 
        '4': 'Ville Monde', 
        '5': 'Région Monde'
    }
    df_locs['Type_Clair'] = df_locs['LocType'].map(type_map).fillna('Inconnu')
    
    # Réorganisation des colonnes pour un affichage propre
    df_locs = df_locs[['Type_Clair', 'CountryCode', 'LocName', 'mentions']]
    
    con.close()
    return df_locs

# ── EXEMPLE D'UTILISATION ──
# Pointez vers UN SEUL fichier de votre base de données brute (pas les indicateurs générés)
fichier_test = "./gdelt_parquet_db/gdelt_2026-05.parquet" # <- À modifier avec votre vrai chemin

if Path(fichier_test).exists():
    df_top_locations = explore_gdelt_locations(fichier_test, limit=60)
    display(df_top_locations)
else:
    print(f"Fichier introuvable : {fichier_test}. Modifiez le chemin.")

Analyse des localisations en cours sur ./gdelt_parquet_db/gdelt_2026-05.parquet ...


,Type_Clair,CountryCode,LocName,mentions
0,Pays,CH,China,3445961
1,Pays,IR,Iran,2743911
2,Pays,US,United States,2112431
3,Pays,RS,Russia,1583906
4,Pays,UP,Ukraine,1443390
5,Pays,IN,India,1354276
6,Pays,UK,United Kingdom,1255006
7,Pays,IS,Israel,1098005
8,Pays,RS,Russian,942217
9,Pays,GM,Germany,871467
